In [1]:
import numpy as np
import pandas as pd
import matplotlib as plt
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn .metrics import accuracy_score, f1_score

## 组成数据表

In [3]:
bank_train = pd.read_csv(r"D:\WeChat Files\wxid_vzlr2hkxrgil22\FileStorage\File\2024-09\2024.9.29\2024.9.29\bank.csv", engine = 'python', encoding='utf_8')
overdue_train = pd.read_csv(r"D:\WeChat Files\wxid_vzlr2hkxrgil22\FileStorage\File\2024-09\overdue.csv", engine = 'python', encoding='utf_8',index_col=0)
bill_train = pd.read_csv(r"D:\WeChat Files\wxid_vzlr2hkxrgil22\FileStorage\File\2024-09\bill.csv", engine = 'python', encoding='utf_8')
userinfo_train = pd.read_csv(r"D:\WeChat Files\wxid_vzlr2hkxrgil22\FileStorage\File\2024-09\userinfo.csv", engine = 'python', encoding='utf_8')

#### 生成状态特征

In [7]:
bill_train["欠款金额"] = bill_train["上期账单金额"] - bill_train["上期还款金额"]
agg_cormoney = bill_train.groupby("new_user_id")["欠款金额"].agg(["mean"])
agg_cormoney.columns = ["平均欠款金额"]
agg_cormoney.reset_index(inplace=True)
agg_cormoney.head(5)

,new_user_id,平均欠款金额
0,0000a11bede26d01990cd7bf86a4ffed,1.595222
1,0001102f53f24a8952ed5b47f9564354,-1.647174
2,000228d848615f1320914e0cf6064d12,-24.352021
3,0003f283dfacd7100bba76d876cf94da,-3.289508
4,000fc975cf7a868719446414a5034eae,0.702195


In [8]:
agg_salary1 = bank_train[bank_train['工资收入标记'] == 1].groupby("new_user_id")['交易金额'].agg(["mean"])
agg_salary1.columns = ['平均工资收入']
agg_salary1.reset_index(inplace=True)
agg_salary1.head(5)

,new_user_id,平均工资收入
0,00375b8a7a62da4f578f4f4464fc4228,44.964550
1,00872649357929f1f3d7fafbe484810b,46.415984
2,00e44c200eeb161df7d08ac1b79bd4a7,47.292812
3,02419b54629f540d3430e6ba8230aac4,45.600048
4,024c12c231362aa4206c001674abe30f,40.380961


##### 合并到总表

In [6]:
# 定义总表
df = pd.merge(userinfo_train, overdue_train, on='new_user_id', how='left')
df = pd.merge(df, agg_cormoney, on='new_user_id', how='left')
df = pd.merge(df, agg_salary1, on='new_user_id', how='left')
df = df.fillna(0)  # 填补空值
df.head(5)

,new_user_id,性别,职业,教育程度,婚姻状态,户口类型,标签,分数,平均欠款金额,平均工资收入
0,7259fe272934e0dc348092c37544a445,女,公职,本科及以上,未婚,集体户口,0,56.575238,0.234447,0.000000
1,0c87f6b1f7e0350a35841ba34df9d26a,男,公职,本科及以上,离异,集体户口,1,85.418075,-11.344397,44.516465
2,74c53a62f15cc039864aefc41170f81b,男,公职,大专,离异,家庭户口,1,79.918908,-4.003015,46.546842
3,211f56f356fa6ecc6334bb73e52c7b16,男,企业,高中,离异,农业户口,0,52.051838,0.000000,0.000000
4,07b0d9ac7924424894141b2fecfa0ee0,男,公职,本科及以上,已婚,集体户口,0,54.877265,0.294647,0.000000


##### 编码

In [7]:
df1 = df[['职业', '教育程度', '婚姻状态', '平均欠款金额', '平均工资收入']]
df1.head(5)
df1.shape
new_df = pd.get_dummies(df1, columns=['职业', '教育程度', '婚姻状态'])
new_df.head(5)

,平均欠款金额,平均工资收入,职业_0,职业_事业单位,职业_企业,职业_公职,职业_学生,职业_无业,教育程度_初中及以下,教育程度_大专,教育程度_未知,教育程度_本科及以上,教育程度_高中,婚姻状态_0,婚姻状态_丧偶,婚姻状态_其他情况,婚姻状态_已婚,婚姻状态_未婚,婚姻状态_未知,婚姻状态_离异
0,0.234447,0.000000,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,True,False,False
1,-11.344397,44.516465,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,True
2,-4.003015,46.546842,False,False,False,True,False,False,False,True,False,False,False,False,False,False,False,False,False,True
3,0.000000,0.000000,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,True
4,0.294647,0.000000,False,False,False,True,False,False,False,False,False,True,False,False,False,False,True,False,False,False


##### 训练与输出

In [8]:
X_train, X_test, Y_train, Y_test = train_test_split(new_df, df[['标签']], test_size=0.2)
model = LogisticRegression()
model.fit(X_train, Y_train)
Y_pred = model.predict(X_test)
print("accuracy score:",accuracy_score(Y_test, Y_pred)) 
print("f1 score:",f1_score(Y_test, Y_pred))

accuracy score: 0.7735
f1 score: 0.3083969465648855


d:\Python\Lib\site-packages\sklearn\utils\validation.py:1339: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
d:\Python\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


#### 生成趋势特征

In [12]:
25*1.6

40.0

In [9]:
def cust_period(value, op="days", op_period=0, base_dt=None):
    value_list = value.tolist()
    
    while "-1" in value_list:
        value_list.remove("-1")
    if len(value_list) != 0:
        if not base_dt:
            max_dt = max(value_list)
        else:
            max_dt = base_dt

        if op == "years":
            end_dt = (datetime.datetime.strptime(max_dt, "%Y-%m-%d %H:%M:%S") + relativedelta(years=-op_period)).strftime("%Y-%m-%d %H:%M:%S")
        elif op == "months":
            end_dt = (datetime.datetime.strptime(max_dt, "%Y-%m-%d %H:%M:%S") + relativedelta(months=-op_period)).strftime("%Y-%m-%d %H:%M:%S")
        elif op == "weeks":
            end_dt = (datetime.datetime.strptime(max_dt, "%Y-%m-%d %H:%M:%S") + relativedelta(weeks=-op_period)).strftime("%Y-%m-%d %H:%M:%S")
        elif op == "days":
            end_dt = (datetime.datetime.strptime(max_dt, "%Y-%m-%d %H:%M:%S") + relativedelta(days=-op_period)).strftime("%Y-%m-%d %H:%M:%S")
        else:
            print("Wrong operation unit of time!")
            exit()
        period = list(value.values >= end_dt)
        return period
    else:
        return [True]*len(value.tolist())


In [11]:
import datetime
from dateutil.relativedelta import relativedelta
agg_agg_6m = bank_train[bank_train['工资收入标记'] == 1].groupby("new_user_id")[["交易金额","交易时间"]]
agg_agg_6m = agg_agg_6m.apply(lambda x: x[cust_period(x["交易时间"], "months", 6)])
agg_agg_6m = agg_agg_6m.groupby("new_user_id")["交易金额"].agg(["sum"])
agg_agg_6m.columns = ["6月内工资收入"]
agg_agg_6m.reset_index(inplace=True)
print(agg_agg_6m.head(5))
agg_agg_3m = bank_train[bank_train['工资收入标记'] == 1].groupby("new_user_id")[["交易金额","交易时间"]]
agg_agg_3m = agg_agg_3m.apply(lambda x: x[cust_period(x["交易时间"], "months", 3)])
agg_agg_3m = agg_agg_3m.groupby("new_user_id")["交易金额"].agg(["sum"])
agg_agg_3m.columns = ["3月内工资收入"]
agg_agg_3m.reset_index(inplace=True)
print(agg_agg_3m.head(5))
agg_tend_3m = pd.merge(agg_agg_6m, agg_agg_3m, how='left', on=['new_user_id'])
agg_tend_3m["半年到3月前工资收入"] = agg_tend_3m["6月内工资收入"] - agg_tend_3m['3月内工资收入']
print(agg_tend_3m.head(5))
def cal(x):
    if x['3月内工资收入'] >= x['半年到3月前工资收入']:
        return 1
    return 0
agg_tend_3m["过去半年内工资增长趋势"] = agg_tend_3m.apply(cal, axis=1)
print(agg_tend_3m.head(5))

                        new_user_id     6月内工资收入
0  00375b8a7a62da4f578f4f4464fc4228  179.858202
1  00872649357929f1f3d7fafbe484810b  232.079920
2  00e44c200eeb161df7d08ac1b79bd4a7   47.292812
3  02419b54629f540d3430e6ba8230aac4  723.807150
4  024c12c231362aa4206c001674abe30f  444.190576
                        new_user_id     3月内工资收入
0  00375b8a7a62da4f578f4f4464fc4228  179.858202
1  00872649357929f1f3d7fafbe484810b  232.079920
2  00e44c200eeb161df7d08ac1b79bd4a7   47.292812
3  02419b54629f540d3430e6ba8230aac4  505.340387
4  024c12c231362aa4206c001674abe30f  240.997538
                        new_user_id     6月内工资收入     3月内工资收入  半年到3月前工资收入
0  00375b8a7a62da4f578f4f4464fc4228  179.858202  179.858202    0.000000
1  00872649357929f1f3d7fafbe484810b  232.079920  232.079920    0.000000
2  00e44c200eeb161df7d08ac1b79bd4a7   47.292812   47.292812    0.000000
3  02419b54629f540d3430e6ba8230aac4  723.807150  505.340387  218.466763
4  024c12c231362aa4206c001674abe30f  444.190576  240.997538  203

In [7]:
bank_salary_statistic = bank[bank['工资收入标记']==1].groupby(['new_user_id'])["交易金额"].agg(["mean", "count"]).reset_index()
bank_no_salary_statistic = bank[bank['工资收入标记']!=1].groupby(['new_user_id'])["交易金额"].agg(["mean", "count"]).reset_index()

In [8]:
type_one_statistic = bank[bank['交易类型']==1].groupby(['new_user_id'])["交易金额"].agg(['mean','count','sum']).reset_index()
type_zero_statistic = bank[bank['交易类型']==0].groupby(['new_user_id'])["交易金额"].agg(['mean','count','sum']).reset_index()

In [9]:
type_zero_statistic

,new_user_id,mean,count,sum
0,0003f283dfacd7100bba76d876cf94da,43.751671,43,1881.321862
1,00284cf15ae27d1ddf4f93922cd7bcb5,46.666179,673,31406.338786
2,00375b8a7a62da4f578f4f4464fc4228,44.783129,191,8553.577580
3,0057bbe021e9c3fc79834e3ff126bb98,45.167214,101,4561.888566
4,00872649357929f1f3d7fafbe484810b,43.622189,18,785.199403
...,...,...,...,...
1677,ff1d4165c6f00864249193f5e3de1670,45.594458,115,5243.362714
1678,ff2d236c45aae847d1a1694520fd38f9,43.963605,112,4923.923722
1679,ff4a7b68f9856efa64c341ada41adb84,46.775009,685,32040.881503
1680,ffc8a37173c1046de20b0f3b6286cd88,44.050161,238,10483.938374


In [10]:
type_one_statistic 

,new_user_id,mean,count,sum
0,0003f283dfacd7100bba76d876cf94da,41.178247,111,4570.785390
1,00284cf15ae27d1ddf4f93922cd7bcb5,45.626841,1327,60546.818365
2,00375b8a7a62da4f578f4f4464fc4228,44.355175,274,12153.317912
3,0057bbe021e9c3fc79834e3ff126bb98,43.405148,266,11545.769269
4,00872649357929f1f3d7fafbe484810b,43.453143,47,2042.297701
...,...,...,...,...
1678,ff1d4165c6f00864249193f5e3de1670,43.084673,348,14993.466328
1679,ff2d236c45aae847d1a1694520fd38f9,42.047611,669,28129.852051
1680,ff4a7b68f9856efa64c341ada41adb84,44.925815,1315,59077.446573
1681,ffc8a37173c1046de20b0f3b6286cd88,42.885300,880,37739.063872


In [11]:
bank_salary_statistic

,new_user_id,mean,count
0,00375b8a7a62da4f578f4f4464fc4228,44.964550,4
1,00872649357929f1f3d7fafbe484810b,46.415984,5
2,00e44c200eeb161df7d08ac1b79bd4a7,47.292812,1
3,02419b54629f540d3430e6ba8230aac4,45.600048,28
4,024c12c231362aa4206c001674abe30f,40.380961,11
...,...,...,...
551,fdfcea9217eac320f5d2e30a58f356fa,42.958105,2
552,fe000bc1c6325b6df010187cca665a1e,46.980840,12
553,fed554907bfead73a15b053ecec59e11,47.292812,2
554,fee3ee14f7b2e42e86746d19115ef14a,47.061820,8


In [13]:
bank_salary_statistic.columns = ['new_user_id', '平均工资收入', '工资交易次数']
bank_no_salary_statistic.columns = ['new_user_id', '平均非工资收入', '非工资交易次数']

In [14]:
trainData = pd.merge(overdue, bank_salary_statistic, on='new_user_id')
trainData = pd.merge(trainData, bank_no_salary_statistic, on='new_user_id')

In [15]:
trainData

,new_user_id,标签,分数,平均工资收入,工资交易次数,平均非工资收入,非工资交易次数
0,7ad5361fd9590ea88dc750aa0274332c,1,85.099537,46.223939,33,43.500662,85
1,b8aba197b170e9d0dcdcd026e7b13cf8,1,81.915179,46.758558,5,43.081445,176
2,7aac1c425037eb6021619dd4daa4b8da,0,77.166123,46.672570,10,44.285470,68
3,f68111fb49fedd89569d22c76375b1dd,1,79.570934,46.364722,4,43.144124,669
4,b6835224c9880d44a9e4b237241ee287,1,81.960435,42.524182,15,44.631923,1152
...,...,...,...,...,...,...,...
551,ac2d4d1ab1fba20a425e2bc3570ab343,1,81.464831,45.925783,3,44.999368,485
552,c979dfde099ebd485bce397ae2ea1666,1,85.925485,45.859023,1,42.731148,18
553,5a977b232c56becd563f19e4e39cae20,1,84.496179,44.623643,1,44.427533,502
554,4f472d052f8e0e9c8e90e849cf1497e7,1,81.140144,46.961614,6,42.864670,119


In [16]:
trainData['工资与非工资收入差值']=trainData['平均工资收入']-trainData['平均非工资收入']
trainData['工资与非工资次数差值']=trainData['工资交易次数']-trainData['平均非工资收入']

In [17]:
trainData.columns

Index(['new_user_id', '标签', '分数', '平均工资收入', '工资交易次数', '平均非工资收入', '非工资交易次数',
       '工资与非工资收入差值', '工资与非工资次数差值'],
      dtype='object')

In [18]:
trainData = trainData[['分数','标签','平均工资收入', '工资交易次数', '平均非工资收入', '非工资交易次数','工资与非工资收入差值', '工资与非工资次数差值']]

In [19]:
trainData.columns = [ 'Y1','Y2','X1','X2','X3','X4','X5','X6']

In [20]:
trainData

,Y1,Y2,X1,X2,X3,X4,X5,X6
0,85.099537,1,46.223939,33,43.500662,85,2.723277,-10.500662
1,81.915179,1,46.758558,5,43.081445,176,3.677113,-38.081445
2,77.166123,0,46.672570,10,44.285470,68,2.387100,-34.285470
3,79.570934,1,46.364722,4,43.144124,669,3.220598,-39.144124
4,81.960435,1,42.524182,15,44.631923,1152,-2.107741,-29.631923
...,...,...,...,...,...,...,...,...
551,81.464831,1,45.925783,3,44.999368,485,0.926415,-41.999368
552,85.925485,1,45.859023,1,42.731148,18,3.127875,-41.731148
553,84.496179,1,44.623643,1,44.427533,502,0.196110,-43.427533
554,81.140144,1,46.961614,6,42.864670,119,4.096945,-36.864670


我们现在就是要训练一个模型，具体来讲，这个模型就是用平均工资收入,工资交易次数,平均非工资收入,非工资交易次数,工资与非工资收入差值,工资与非工资次数差值来预测分数值：

$Y = f(X)$

# 根据原理写多元线性回归

In [21]:
from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(trainData[['X1','X2','X3','X4']], trainData[['Y1']],test_size=0.25,random_state=2022)

In [22]:
X_train

,X1,X2,X3,X4
315,45.657506,5,43.560124,2586
106,46.848852,15,44.235547,991
464,45.420179,13,42.839888,246
169,44.147992,7,44.213699,1120
540,46.581155,16,43.733239,512
...,...,...,...,...
24,45.899144,16,42.639653,1586
18,47.647217,2,43.135811,158
240,46.903288,2,42.727610,502
173,43.332568,8,43.344084,752


In [23]:
#归一化：把数据转换为（0,1）区间的小数，  主要是为了数据处理方便提出来的，把数据映射到0～1范围之内处理，更加便捷快速。
X_train = (X_train-X_train.min())/(X_train.max()-X_train.min())
X_test = (X_test-X_test.min())/(X_test.max()-X_test.min())

In [24]:
X_train

,X1,X2,X3,X4
315,0.797338,0.008163,0.583022,1.000000
106,0.878791,0.028571,0.690823,0.382262
464,0.781112,0.024490,0.468068,0.093726
169,0.694133,0.012245,0.687336,0.432223
540,0.860488,0.030612,0.610652,0.196747
...,...,...,...,...
24,0.813859,0.030612,0.436109,0.612703
18,0.933375,0.002041,0.515299,0.059644
240,0.882513,0.002041,0.450148,0.192874
173,0.638382,0.014286,0.548541,0.289698


In [25]:
X_train = np.array(X_train)
Y_train = np.array(Y_train)
X_test = np.array(X_test)
Y_test = np.array(Y_test)

In [26]:

a = np.array([[1, 1],
              [2, 2],
              [3, 3]])
b = np.array([[4, 4],
              [5, 5],
              [6, 6]])


In [27]:
np.vstack((a,b))

array([[1, 1],
       [2, 2],
       [3, 3],
       [4, 4],
       [5, 5],
       [6, 6]])

In [28]:
a = np.array([[1, 1],
              [2, 2],
              [3, 3]])
a.transpose()

array([[1, 2, 3],
       [1, 2, 3]])

In [29]:
# x=[x1,x2,x3,x4,1]
# t =[theta1,theta2,theta3,theta4,theta0]

x = np.array([
    [1, 2, 3, 4, 1],
    [3, 4, 2, 2, 1]
])
t = np.array([1,1,1,1,1])
np.dot(x,t)


array([11, 12])

In [30]:
#我们有两个变量，为了对这个公式进行矩阵化，我们可以给每一个点x增加一维，这一维的值固定为1，这一维将会乘到Θ0上。这样就方便我们统一矩阵化的计算
train_m=len(X_train)
X_train_marixadd=np.ones((train_m,1))
X_train=np.hstack((X_train,X_train_marixadd))

In [31]:
#我们有两个变量，为了对这个公式进行矩阵化，我们可以给每一个点x增加一维，这一维的值固定为1，这一维将会乘到Θ0上。这样就方便我们统一矩阵化的计算
test_m=len(X_test)
X_test_marixadd=np.ones((test_m,1))
X_test=np.hstack((X_test,X_test_marixadd))

In [32]:
#梯度计算函数
def gradient_function(theta,X,Y,m):
    temp = np.dot(X,theta)-Y
    return (1/m)*np.dot(X.transpose(),temp)

#迭代函数
def gradient_descent(X,Y,alpha,m):
    #定义最早的theta为<1,1,1,1,1>矩阵，并转换为列向量
    theta = np.array([1,1,1,1,1]).reshape(5,1)
    #获取第一个梯度
    gradient = gradient_function(theta,X,Y,m)
    #迭代更新，规定当梯度小于10**-5时停止迭代，并返回此时的theta
    while not all(abs(gradient) <= 10**-5):
        theta = theta - alpha * gradient
        gradient = gradient_function(theta,X,Y,m)
        #print(gradient)
    return theta

In [33]:
from sklearn.metrics import mean_squared_error, r2_score # 评估指标
alpha = 1e-4
optimal = gradient_descent(X_train,Y_train,alpha,train_m)
#print(optimal)
Y_pred = np.dot(X_test,optimal)
print('Mean squared error: %.2f' % mean_squared_error(Y_test, Y_pred)) #预测均方差
print('Coefficient of determination: %.2f' % r2_score(Y_test, Y_pred))  #r^2

Mean squared error: 15.25
Coefficient of determination: -0.04


# scikit-learn 的应用

scikit-learn 的应用
好了，这里必须明确一个重点：

**所有的 sklearn 用法都差不多，因此这里要看懂。**

**所有的 sklearn 用法都差不多，因此这里要看懂。**

**所有的 sklearn 用法都差不多，因此这里要看懂。**

sklearn 训练流程如下：

1. 整理好你的训练数据X_train和Y_train以及你的测试数据X_test和Y_test
2. 定义你要的模型model
3. 让你的数据训练你的模型model.fit(X_train, Y_train)
4. 使用model.predict(X_test)看 f(X) 的值

# 多元线性回归

In [34]:
from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(trainData[['X1','X2','X3','X4']], trainData[['Y1']],test_size=0.25,random_state=2022)

In [35]:
X_train = (X_train-X_train.min())/(X_train.max()-X_train.min())
X_test = (X_test-X_test.min())/(X_test.max()-X_test.min())

In [36]:
from sklearn import linear_model  # 我们用线性回归模型
from sklearn.metrics import mean_squared_error, r2_score # 评估指标

model = linear_model.LinearRegression() # 定义你要的模型model

model.fit(X_train,Y_train) # 让你的数据训练你的模型model.fit(X_train, Y_train)

Y_pred = model.predict(X_test) # 使用 model.predict(X_test) 看 𝑓(𝑋) 的值

print('Coefficients: \n', model.coef_, model.intercept_) # 打印参数即前面讲的𝜃 分别是线性系数和截距
print('Mean squared error: %.2f' % mean_squared_error(Y_test, Y_pred)) #预测均方差
print('Coefficient of determination: %.2f' % r2_score(Y_test, Y_pred))  #r^2

Coefficients: 
 [[ 3.04493182 13.09473603  3.22792482  2.24008642]] [76.75285437]
Mean squared error: 15.25
Coefficient of determination: -0.04


# 逻辑回归

In [37]:
from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(trainData[['X1','X2','X3','X4']], trainData[['Y2']],test_size=0.25,random_state=2022)

In [38]:
X_test

,X1,X2,X3,X4
466,41.315890,2,41.807906,601
352,46.330034,13,44.670616,281
428,46.026259,12,42.610478,834
6,45.463252,14,44.406701,333
497,46.355851,11,44.464359,147
...,...,...,...,...
178,46.937818,6,42.297395,219
430,42.107695,8,42.843786,824
404,46.981325,6,44.252658,1821
162,43.052796,39,41.100323,218


In [39]:
# from sklearn import linear_model  # 我们用线性回归模型
# from sklearn.metrics import mean_squared_error, r2_score # 评估指标
from sklearn.metrics import confusion_matrix

# model = linear_model.LogisticRegression() # 定义你要的模型model
model = linear_model.LogisticRegression(class_weight='balanced') # 设置类权重超参数

model.fit(X_train,Y_train) # 让你的数据训练你的模型model.fit(X_train, Y_train)

Y_pred = model.predict(X_test) # 使用 model.predict(X_test) 看 𝑓(𝑋) 的值

cm = confusion_matrix(Y_test, Y_pred)

cm

# print('Coefficients: \n', model.coef_, model.intercept_) # 打印参数即前面讲的𝜃 分别是线性系数和截距
# print('Mean squared error: %.2f' % mean_squared_error(Y_pred, Y_test)) #预测均方差
# print('Coefficient of determination: %.2f' % r2_score(Y_pred, Y_test))  #r^2

C:\Users\lenovo\anaconda3\lib\site-packages\sklearn\utils\validation.py:760: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


array([[ 5,  8],
       [56, 70]], dtype=int64)